# Word Embeddings Demo

This notebook accompanies the cleaned 2018 NLP note in `README.md`.

It walks through the core representational shift:

1. symbolic lexical relations with WordNet
2. one-hot vectors as word IDs
3. context/co-occurrence vectors
4. LSA-style dense vectors via truncated SVD

The point is conceptual clarity, not production NLP.

## Setup

The env script downloads WordNet into a project-local `nltk_data/` folder. If you run this notebook in a different environment, uncomment the download lines below.

In [ ]:
from pathlib import Path

import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

import nltk
from nltk.corpus import wordnet as wn

PROJECT_DIR = Path.cwd()
LOCAL_NLTK_DATA = PROJECT_DIR / 'nltk_data'
if LOCAL_NLTK_DATA.exists():
    nltk.data.path.insert(0, str(LOCAL_NLTK_DATA))

# Uncomment if WordNet has not been downloaded yet.
# nltk.download('wordnet', download_dir=str(LOCAL_NLTK_DATA))
# nltk.download('omw-1.4', download_dir=str(LOCAL_NLTK_DATA))

np.set_printoptions(precision=3, suppress=True)

## 1. WordNet: Words As Explicit Symbolic Relations

WordNet is useful because it exposes human-curated relations between word senses. This is meaning as a graph, not meaning as learned geometry.

In [ ]:
panda = wn.synset('panda.n.01')
hypernym_path = [synset.name() for synset in panda.closure(lambda synset: synset.hypernyms())]

print('Definition:', panda.definition())
print('Hypernym path:')
for item in hypernym_path[:12]:
    print('  ', item)

## 2. One-Hot Vectors: Useful IDs, Bad Similarity

One-hot vectors are convenient for indexing, but they make every distinct word orthogonal to every other distinct word.

In [ ]:
words = ['cat', 'kitten', 'dog', 'couch']
vocab = {word: idx for idx, word in enumerate(words)}

one_hot = np.eye(len(words))

for word in words:
    print(f'{word:7}', one_hot[vocab[word]].astype(int))

print()
print('Cosine similarity under one-hot encoding:')
for other in ['kitten', 'dog', 'couch']:
    sim = cosine_similarity(
        one_hot[vocab['cat']].reshape(1, -1),
        one_hot[vocab[other]].reshape(1, -1),
    )[0, 0]
    print(f'cat vs {other:7}: {sim:.1f}')

## 3. Distributional Similarity: Meaning From Neighbors

Now build simple context vectors from a tiny corpus. This is the old distributional hypothesis in toy form: words with similar neighbors should land near each other.

In [ ]:
corpus = [
    'cat eats fish',
    'kitten eats fish',
    'dog eats food',
    'puppy eats food',
    'cat sleeps on sofa',
    'kitten sleeps on blanket',
    'dog sleeps on rug',
    'sofa sits in living room',
    'couch sits in living room',
    'chair sits in living room',
    'car drives on road',
    'truck drives on road',
]

vectorizer = CountVectorizer(token_pattern=r'(?u)\b\w+\b')
term_doc = vectorizer.fit_transform(corpus)
terms = np.array(vectorizer.get_feature_names_out())

# A term-term co-occurrence matrix can be approximated by multiplying the
# term-document matrix by its transpose. This tiny example counts words that
# appear in the same short sentence.
term_term = (term_doc.T @ term_doc).astype(float)
term_term.setdiag(0)
cooc_vectors = normalize(term_term, norm='l2', axis=1)

term_to_idx = {term: idx for idx, term in enumerate(terms)}

print('Vocabulary:', terms.tolist())
print()
print('Cosine similarity under co-occurrence vectors:')
for other in ['kitten', 'dog', 'couch', 'car']:
    sim = cosine_similarity(
        cooc_vectors[term_to_idx['cat']],
        cooc_vectors[term_to_idx[other]],
    )[0, 0]
    print(f'cat vs {other:7}: {sim:.3f}')

## 4. Dense Vectors With Truncated SVD

LSA uses truncated SVD to project sparse term/document or term/context patterns into a smaller dense space. This is not Word2Vec, but it teaches the same geometric intuition: words can be placed in a learned semantic space.

In [ ]:
n_components = 2
svd = TruncatedSVD(n_components=n_components, random_state=7)
dense_vectors = svd.fit_transform(term_term)
dense_vectors = normalize(dense_vectors, norm='l2', axis=1)

for word in ['cat', 'kitten', 'dog', 'puppy', 'couch', 'sofa', 'car', 'truck']:
    vec = dense_vectors[term_to_idx[word]]
    print(f'{word:7} -> [{vec[0]: .3f}, {vec[1]: .3f}]')

print()
print('Cosine similarity in the dense SVD space:')
for other in ['kitten', 'dog', 'couch', 'car']:
    sim = cosine_similarity(
        dense_vectors[term_to_idx['cat']].reshape(1, -1),
        dense_vectors[term_to_idx[other]].reshape(1, -1),
    )[0, 0]
    print(f'cat vs {other:7}: {sim:.3f}')

## 5. Inspect The Nearest Neighbors

With a learned vector space, similarity becomes a retrieval operation.

In [ ]:
def nearest_neighbors(word, vectors, top_k=5):
    idx = term_to_idx[word]
    sims = cosine_similarity(vectors[idx].reshape(1, -1), vectors).ravel()
    order = np.argsort(-sims)
    return [(terms[i], float(sims[i])) for i in order if terms[i] != word][:top_k]

for word in ['cat', 'dog', 'couch', 'car']:
    print(word)
    for neighbor, score in nearest_neighbors(word, dense_vectors):
        print(f'  {neighbor:8} {score:.3f}')

## 6. Where Transformers Change The Story

Word2Vec-style vectors are mostly fixed for a word type. A transformer representation is contextual: the vector for `bank` changes depending on whether the sentence is about money or a river.

That makes transformer embeddings more flexible, but the conceptual inheritance is direct. Modern NLP still lives in vector spaces; it just builds those vectors from richer context.